In [ ]:
# =============================================================
# 03_create_masks_and_labels.ipynb
# Step 3
# ------------------------------------------------------------
# What this does
# ──────────────
# Reads the wide panel produced by Step 2 and generates:
#
# 1. Trajectory labels (4.2)
#    Four-class typology based on two decade-long deltas:
#      ΔGini        ≤ −0.5 pp  →  inequality declining
#      ΔTertiary    ≥ +2.0 pp  →  higher-education expanding
#
#    Classes:
#      Equity Expander   (EE)  — both conditions met
#      Access Expander   (AE)  — tertiary ↑ only
#      Inequality Reducer(IR)  — Gini ↓ only
#      Data-Limited      (DL)  — severe Gini missingness
#                                (coverage < DL_GINI_THRESHOLD)
#
#    Labels are assigned at country level (one per country).
#    Saved to: data/processed/trajectory_labels.csv
#
# 2. Train/test split indices (5. evaluation protocol)
#    Two validation configurations:
#
#    a) Temporal split
#       Train: 2015–2021  (years 1–7)
#       Test:  2022–2024  (years 8–10)
#       Saved to: data/processed/train_test_splits/temporal_split.csv
#
#    b) Leave-One-Country-Out (LOCO)
#       One fold per country: held-out country = test,
#       remaining 11 countries = train.
#       Saved to: data/processed/train_test_splits/loco_splits.csv
#
# 3. Delta summary table
#    Per-country ΔGini and ΔTertiary with labelling rationale.
#    Saved to: data/processed/trajectory_delta_summary.csv
#
# Usage
# ─────
#   python src/03_create_masks_and_labels.py
#
# Requirements
# ────────────
#   pip install pandas numpy matplotlib pyyaml
# =============================================================

from __future__ import annotations

import sys
from pathlib import Path

# Notebook-safe path setup
try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    _SRC = _cwd / "src" if (_cwd / "src").exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from utils import find_project_root, load_pipeline, INCOME_ORDER


# ──────────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"

# Trajectory label thresholds (§4.2)
DELTA_GINI_THRESHOLD   = -0.5   # pp — Gini must fall by at least this
DELTA_TERT_THRESHOLD   = +2.0   # pp — tertiary enrolment must rise by at least this
DL_GINI_THRESHOLD      = 0.20   # coverage share — below this → Data-Limited

# Temporal split years
TRAIN_END = 2021
TEST_START = 2022

DATA_DIR   = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR  = DATA_DIR / "train_test_splits"
FIGDIR     = PROJECT_ROOT / "outputs" / "figures" / "panel_12"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

print(f"[info] Project root  : {PROJECT_ROOT}")
print(f"[info] Year range    : {YEAR_MIN}–{YEAR_MAX}")
print(f"[info] Train window  : {YEAR_MIN}–{TRAIN_END}")
print(f"[info] Test window   : {TEST_START}–{YEAR_MAX}")
print(f"[info] ΔGini thresh  : {DELTA_GINI_THRESHOLD} pp")
print(f"[info] ΔTert thresh  : +{DELTA_TERT_THRESHOLD} pp")
print(f"[info] DL Gini cov   : < {DL_GINI_THRESHOLD:.0%}")


# ──────────────────────────────────────────────────────────────
# Load panel
# ──────────────────────────────────────────────────────────────
def load_panel() -> pd.DataFrame:
    p = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.csv"
    if not p.exists():
        raise FileNotFoundError(
            f"Panel not found: {p}\n"
            "Run 02_clean_merge_panel.py first."
        )
    panel = pd.read_csv(p)
    print(f"\n[step 0] Loaded panel: {len(panel)} rows × {len(panel.columns)} columns")
    return panel


# ──────────────────────────────────────────────────────────────
# Step 1 — Compute decade-long deltas
# ──────────────────────────────────────────────────────────────
def compute_deltas(panel: pd.DataFrame) -> pd.DataFrame:
    """
    For each country, compute:
      ΔGini     = Gini(latest observed) − Gini(earliest observed)
      ΔTertiary = TertEnrol(latest) − TertEnrol(earliest)
      Gini_coverage = fraction of 10 years with non-null Gini

    Uses first and last non-null values across the panel window,
    not anchored to specific years, to maximise use of available data.
    """
    records = []
    for iso, grp in panel.groupby("iso3c"):
        grp = grp.sort_values("year")
        name   = grp["country"].iloc[0]
        tier   = grp["income_group"].iloc[0]

        # Gini delta
        gini_obs = grp["gini"].dropna()
        gini_cov = grp["gini"].notna().mean()
        if len(gini_obs) >= 2:
            delta_gini  = float(gini_obs.iloc[-1] - gini_obs.iloc[0])
            gini_first  = float(gini_obs.iloc[0])
            gini_last   = float(gini_obs.iloc[-1])
            gini_yr_first = int(grp.loc[grp["gini"].notna(), "year"].iloc[0])
            gini_yr_last  = int(grp.loc[grp["gini"].notna(), "year"].iloc[-1])
        else:
            delta_gini  = np.nan
            gini_first  = np.nan
            gini_last   = np.nan
            gini_yr_first = None
            gini_yr_last  = None

        # Tertiary enrolment delta
        tert_obs = grp["tert_enrol_gross"].dropna()
        if len(tert_obs) >= 2:
            delta_tert  = float(tert_obs.iloc[-1] - tert_obs.iloc[0])
            tert_first  = float(tert_obs.iloc[0])
            tert_last   = float(tert_obs.iloc[-1])
            tert_yr_first = int(grp.loc[grp["tert_enrol_gross"].notna(), "year"].iloc[0])
            tert_yr_last  = int(grp.loc[grp["tert_enrol_gross"].notna(), "year"].iloc[-1])
        else:
            delta_tert  = np.nan
            tert_first  = np.nan
            tert_last   = np.nan
            tert_yr_first = None
            tert_yr_last  = None

        # Completion delta (for reporting)
        comp_obs = grp["sec_completion"].dropna()
        delta_comp = float(comp_obs.iloc[-1] - comp_obs.iloc[0]) \
                     if len(comp_obs) >= 2 else np.nan

        records.append({
            "iso3c":          iso,
            "country":        name,
            "income_group":   tier,
            "gini_coverage":  round(gini_cov, 3),
            "gini_first":     gini_first,
            "gini_first_yr":  gini_yr_first,
            "gini_last":      gini_last,
            "gini_last_yr":   gini_yr_last,
            "delta_gini":     round(delta_gini, 3) if not np.isnan(delta_gini) else np.nan,
            "tert_first":     tert_first,
            "tert_first_yr":  tert_yr_first,
            "tert_last":      tert_last,
            "tert_last_yr":   tert_yr_last,
            "delta_tert":     round(delta_tert, 3) if not np.isnan(delta_tert) else np.nan,
            "delta_comp":     round(delta_comp, 3) if not np.isnan(delta_comp) else np.nan,
        })

    deltas = pd.DataFrame(records)
    deltas = (
        deltas
        .assign(_isort=lambda d: d["income_group"].map(INCOME_ORDER).fillna(4))
        .sort_values(["_isort", "iso3c"])
        .drop(columns="_isort")
        .reset_index(drop=True)
    )
    return deltas


# ──────────────────────────────────────────────────────────────
# Step 2 — Assign trajectory labels
# ──────────────────────────────────────────────────────────────
CLASS_CODES = {
    "EE": "Equity Expander",
    "AE": "Access Expander",
    "IR": "Inequality Reducer",
    "DL": "Data-Limited",
}

CLASS_COLOURS = {
    "EE": "#1a7a3a",   # green
    "AE": "#1a6fad",   # blue
    "IR": "#d73027",   # red
    "DL": "#888888",   # grey
}


def assign_labels(deltas: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the four-class typology rules in priority order:

    Priority 1 — Data-Limited:
      Gini coverage < DL_GINI_THRESHOLD  →  DL
      (Assigned first regardless of delta values.)

    Priority 2 — Equity Expander:
      ΔGini ≤ DELTA_GINI_THRESHOLD  AND  ΔTertiary ≥ DELTA_TERT_THRESHOLD  →  EE

    Priority 3 — Access Expander:
      ΔTertiary ≥ DELTA_TERT_THRESHOLD  (Gini not declining sufficiently)  →  AE

    Priority 4 — Inequality Reducer:
      ΔGini ≤ DELTA_GINI_THRESHOLD  (tertiary not expanding sufficiently)  →  IR

    Residual — Equity Expander (default for any country with both deltas
      available but neither threshold met):  labelled AE by default since
      tertiary growth is ubiquitous; reviewer can override.
    """
    labels = []
    reasons = []

    for _, row in deltas.iterrows():
        gc   = row["gini_coverage"]
        dg   = row["delta_gini"]
        dt   = row["delta_tert"]
        iso  = row["iso3c"]

        # Priority 1 — Data-Limited
        if gc < DL_GINI_THRESHOLD:
            labels.append("DL")
            reasons.append(
                f"Gini coverage {gc:.0%} < {DL_GINI_THRESHOLD:.0%} threshold"
            )
            continue

        gini_declining = (not np.isnan(dg)) and (dg <= DELTA_GINI_THRESHOLD)
        tert_expanding = (not np.isnan(dt)) and (dt >= DELTA_TERT_THRESHOLD)

        # Priority 2 — Equity Expander
        if gini_declining and tert_expanding:
            labels.append("EE")
            reasons.append(
                f"ΔGini={dg:+.1f} pp ≤ {DELTA_GINI_THRESHOLD} pp  AND  "
                f"ΔTert={dt:+.1f} pp ≥ +{DELTA_TERT_THRESHOLD} pp"
            )
        # Priority 3 — Access Expander
        elif tert_expanding:
            labels.append("AE")
            reasons.append(
                f"ΔTert={dt:+.1f} pp ≥ +{DELTA_TERT_THRESHOLD} pp  "
                f"(ΔGini={dg:+.1f} pp, not declining sufficiently)"
                if not np.isnan(dg) else
                f"ΔTert={dt:+.1f} pp ≥ +{DELTA_TERT_THRESHOLD} pp  "
                f"(Gini data insufficient)"
            )
        # Priority 4 — Inequality Reducer
        elif gini_declining:
            labels.append("IR")
            reasons.append(
                f"ΔGini={dg:+.1f} pp ≤ {DELTA_GINI_THRESHOLD} pp  "
                f"(ΔTert={dt:+.1f} pp, not expanding sufficiently)"
                if not np.isnan(dt) else
                f"ΔGini={dg:+.1f} pp ≤ {DELTA_GINI_THRESHOLD} pp  "
                f"(tertiary data insufficient)"
            )
        # Residual
        else:
            dg_str = f"{dg:+.1f}" if not np.isnan(dg) else "n/a"
            dt_str = f"{dt:+.1f}" if not np.isnan(dt) else "n/a"
            labels.append("AE")
            reasons.append(
                f"Residual (ΔGini={dg_str} pp, ΔTert={dt_str} pp — "
                "neither threshold met; default Access Expander)"
            )

    deltas = deltas.copy()
    deltas["label_code"]  = labels
    deltas["label_name"]  = [CLASS_CODES[l] for l in labels]
    deltas["label_reason"] = reasons
    return deltas


# ──────────────────────────────────────────────────────────────
# Step 3 — Build train / test split indices
# ──────────────────────────────────────────────────────────────
def build_temporal_split(panel: pd.DataFrame) -> pd.DataFrame:
    """
    Temporal split:
      train = YEAR_MIN … TRAIN_END
      test  = TEST_START … YEAR_MAX

    Returns long-format DataFrame with columns:
      iso3c, country, year, split  ('train' | 'test')
    """
    split = panel[["iso3c", "country", "year"]].copy()
    split["split"] = np.where(split["year"] <= TRAIN_END, "train", "test")
    return split.sort_values(["iso3c", "year"]).reset_index(drop=True)


def build_loco_splits(panel: pd.DataFrame) -> pd.DataFrame:
    """
    Leave-One-Country-Out (LOCO):
      For each fold k (k = 0 … 11):
        test  = country k
        train = all other 11 countries

    Returns long-format DataFrame with columns:
      fold, held_out_iso3c, held_out_country, iso3c, country, year, split
    """
    countries = sorted(panel["iso3c"].unique())
    records   = []

    for k, held_out in enumerate(countries):
        held_name = panel.loc[panel["iso3c"] == held_out, "country"].iloc[0]
        for _, row in panel.iterrows():
            records.append({
                "fold":              k,
                "held_out_iso3c":    held_out,
                "held_out_country":  held_name,
                "iso3c":             row["iso3c"],
                "country":           row["country"],
                "year":              row["year"],
                "split":             "test" if row["iso3c"] == held_out else "train",
            })

    loco = pd.DataFrame(records)
    return loco.sort_values(["fold", "iso3c", "year"]).reset_index(drop=True)


# ──────────────────────────────────────────────────────────────
# Step 4 — Visualisations
# ──────────────────────────────────────────────────────────────
def plot_trajectory_scatter(
    deltas: pd.DataFrame,
    labels: pd.DataFrame,
) -> None:
    """
    Scatter plot: ΔGini (x) vs ΔTertiary (y), coloured by trajectory class.
    Threshold lines at ΔGini = −0.5 pp and ΔTert = +2.0 pp.
    """
    fig, ax = plt.subplots(figsize=(9, 7))

    # Threshold lines
    ax.axvline(DELTA_GINI_THRESHOLD, color="#aaaaaa", lw=1.2, ls="--", zorder=1)
    ax.axhline(DELTA_TERT_THRESHOLD, color="#aaaaaa", lw=1.2, ls="--", zorder=1)

    # Quadrant shading
    xlim = (-12, 12)
    ylim = (-15, 55)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    shade_alpha = 0.06
    ax.fill_betweenx([DELTA_TERT_THRESHOLD, ylim[1]],
                     xlim[0], DELTA_GINI_THRESHOLD,
                     color=CLASS_COLOURS["EE"], alpha=shade_alpha, zorder=0)
    ax.fill_betweenx([DELTA_TERT_THRESHOLD, ylim[1]],
                     DELTA_GINI_THRESHOLD, xlim[1],
                     color=CLASS_COLOURS["AE"], alpha=shade_alpha, zorder=0)
    ax.fill_betweenx([ylim[0], DELTA_TERT_THRESHOLD],
                     xlim[0], DELTA_GINI_THRESHOLD,
                     color=CLASS_COLOURS["IR"], alpha=shade_alpha, zorder=0)

    # Quadrant labels
    for txt, x, y, c in [
        ("Equity\nExpander",   xlim[0] + 0.5, ylim[1] - 3,  CLASS_COLOURS["EE"]),
        ("Access\nExpander",   xlim[1] - 2.5, ylim[1] - 3,  CLASS_COLOURS["AE"]),
        ("Inequality\nReducer",xlim[0] + 0.5, ylim[0] + 1,  CLASS_COLOURS["IR"]),
    ]:
        ax.text(x, y, txt, fontsize=8, color=c, alpha=0.6, va="top")

    # Scatter points
    for _, row in labels.iterrows():
        iso  = row["iso3c"]
        dg   = row["delta_gini"]
        dt   = row["delta_tert"]
        lc   = row["label_code"]
        col  = CLASS_COLOURS.get(lc, "#888888")

        if np.isnan(dg) or np.isnan(dt):
            # DL countries: plot at edge with marker
            ax.scatter(-11, -12, color=col, s=120, zorder=4,
                       marker="X", edgecolors="white", linewidths=0.8)
            ax.annotate(
                iso,
                xy=(-11, -12),
                xytext=(-10, -12 + np.random.uniform(-2, 2)),
                fontsize=8, color=col, fontweight="bold",
                arrowprops=dict(arrowstyle="-", color=col, lw=0.6),
            )
        else:
            ax.scatter(dg, dt, color=col, s=140, zorder=4,
                       edgecolors="white", linewidths=1.0,
                       path_effects=[pe.withStroke(linewidth=2.5,
                                     foreground="white")])
            ax.annotate(
                iso,
                xy=(dg, dt),
                xytext=(dg + 0.3, dt + 0.8),
                fontsize=8.5, fontweight="bold", color=col,
                path_effects=[pe.withStroke(linewidth=2.0,
                               foreground="white")],
            )

    # Legend
    from matplotlib.patches import Patch
    legend_handles = [
        Patch(color=CLASS_COLOURS[k], label=f"{k} — {v}")
        for k, v in CLASS_CODES.items()
    ]
    ax.legend(handles=legend_handles, fontsize=9, loc="upper right",
              frameon=True, framealpha=0.92, edgecolor="#cccccc")

    ax.set_xlabel(f"ΔGini index (pp, {YEAR_MIN}–{YEAR_MAX})", fontsize=11)
    ax.set_ylabel(f"ΔTertiary enrolment (pp, {YEAR_MIN}–{YEAR_MAX})", fontsize=11)
    ax.set_title(
        "Trajectory classification — 12-country panel\n"
        f"Thresholds: ΔGini ≤ {DELTA_GINI_THRESHOLD} pp  |  "
        f"ΔTertiary ≥ +{DELTA_TERT_THRESHOLD} pp",
        fontsize=11, fontweight="bold",
    )
    ax.grid(alpha=0.2, lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    out = FIGDIR / f"trajectory_scatter_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_split_overview(
    temporal: pd.DataFrame,
    loco:     pd.DataFrame,
    panel:    pd.DataFrame,
) -> None:
    """
    Two-panel figure:
      Left  — temporal split heatmap (country × year, colour = train/test)
      Right — LOCO fold structure (fold × country, colour = train/test)
    """
    countries = (
        panel.groupby("iso3c")["income_group"].first()
        .reset_index()
        .assign(_isort=lambda d: d["income_group"].map(INCOME_ORDER).fillna(4))
        .sort_values(["_isort", "iso3c"])["iso3c"]
        .tolist()
    )
    iso_to_name = dict(zip(panel["iso3c"], panel["country"]))
    years       = sorted(panel["year"].unique())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                                    gridspec_kw={"wspace": 0.35})

    # ── Left: temporal split ─────────────────────────────────
    grid_t = np.zeros((len(countries), len(years)))
    for i, iso in enumerate(countries):
        for j, yr in enumerate(years):
            row = temporal[(temporal["iso3c"] == iso) & (temporal["year"] == yr)]
            if not row.empty:
                grid_t[i, j] = 1 if row["split"].iloc[0] == "train" else 2

    cmap_t = matplotlib.colors.ListedColormap(["#f0f0f0", "#4393c3", "#d6604d"])
    ax1.imshow(grid_t, aspect="auto", interpolation="nearest",
               vmin=0, vmax=2, cmap=cmap_t)
    ax1.set_title("Temporal split\n(blue=train, red=test)", fontsize=11,
                  fontweight="bold")
    ax1.set_xticks(range(len(years)))
    ax1.set_xticklabels([str(y) for y in years], rotation=45, ha="right",
                        fontsize=8)
    ax1.set_yticks(range(len(countries)))
    ax1.set_yticklabels(
        [f"{iso} ({iso_to_name.get(iso, iso)[:12]})" for iso in countries],
        fontsize=8,
    )
    ax1.set_xlabel("Year", fontsize=10)
    # Train/test boundary line
    boundary = years.index(TEST_START) - 0.5
    ax1.axvline(boundary, color="black", lw=2.0, ls="-")
    ax1.text(boundary - 5, -0.8, f"Train\n≤{TRAIN_END}", ha="right",
             fontsize=8, color="#4393c3")
    ax1.text(boundary + 0.2, -0.8, f"Test\n≥{TEST_START}", ha="left",
             fontsize=8, color="#d6604d")

    # ── Right: LOCO fold structure ───────────────────────────
    folds = sorted(loco["fold"].unique())
    grid_l = np.zeros((len(folds), len(countries)))
    for f in folds:
        fold_data = loco[loco["fold"] == f]
        held_iso  = fold_data["held_out_iso3c"].iloc[0]
        for j, iso in enumerate(countries):
            grid_l[f, j] = 2 if iso == held_iso else 1

    ax2.imshow(grid_l, aspect="auto", interpolation="nearest",
               vmin=0, vmax=2, cmap=cmap_t)
    ax2.set_title("LOCO folds\n(blue=train, red=held-out test)", fontsize=11,
                  fontweight="bold")
    ax2.set_xticks(range(len(countries)))
    ax2.set_xticklabels(countries, rotation=45, ha="right", fontsize=9)
    ax2.set_yticks(range(len(folds)))
    ax2.set_yticklabels([f"Fold {f}" for f in folds], fontsize=8)
    ax2.set_xlabel("Country", fontsize=10)
    ax2.set_ylabel("Fold (held-out country)", fontsize=10)

    fig.suptitle(
        "Benchmark evaluation splits — temporal and LOCO",
        fontsize=13, fontweight="bold", y=1.02,
    )
    fig.tight_layout()
    out = FIGDIR / f"evaluation_splits_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


# ──────────────────────────────────────────────────────────────
# Step 5 — Print summary and save
# ──────────────────────────────────────────────────────────────
def print_label_summary(labels: pd.DataFrame) -> None:
    print(f"\n{'='*65}")
    print("Trajectory class assignments")
    print(f"{'='*65}")
    print(f"  {'ISO':<5} {'Country':<22} {'Tier':<12} {'Class':<25} {'ΔGini':>7}  {'ΔTert':>7}")
    print(f"  {'-'*5} {'-'*22} {'-'*12} {'-'*25} {'-'*7}  {'-'*7}")

    for _, row in labels.iterrows():
        dg_str = f"{row['delta_gini']:+.1f}" if pd.notna(row["delta_gini"]) else "  n/a "
        dt_str = f"{row['delta_tert']:+.1f}" if pd.notna(row["delta_tert"]) else "  n/a "
        tier_short = (row["income_group"]
                      .replace("Upper middle income", "UpMid")
                      .replace("Lower middle income", "LoMid")
                      .replace("High income", "High")
                      .replace("Low income", "Low"))
        print(f"  {row['iso3c']:<5} {row['country']:<22} {tier_short:<12} "
              f"{row['label_code']} {row['label_name']:<20} {dg_str:>7}  {dt_str:>7}")

    print(f"\n  Class counts:")
    for code, name in CLASS_CODES.items():
        n = (labels["label_code"] == code).sum()
        isos = ", ".join(labels.loc[labels["label_code"] == code, "iso3c"].tolist())
        print(f"    {code} {name:<22}: n={n}  ({isos})")
    print(f"{'='*65}")


def save_outputs(
    labels:   pd.DataFrame,
    temporal: pd.DataFrame,
    loco:     pd.DataFrame,
) -> None:
    print("\n[step 5] Saving outputs …")

    # Trajectory labels
    label_cols = ["iso3c", "country", "income_group",
                  "label_code", "label_name",
                  "gini_coverage", "delta_gini", "delta_tert", "delta_comp",
                  "gini_first", "gini_first_yr", "gini_last", "gini_last_yr",
                  "tert_first", "tert_first_yr", "tert_last", "tert_last_yr",
                  "label_reason"]
    p1 = DATA_DIR / f"trajectory_labels.csv"
    labels[label_cols].to_csv(p1, index=False)
    print(f"  [OK] {p1.name}  ({len(labels)} countries)")

    # Delta summary (concise version for paper table)
    p2 = DATA_DIR / f"trajectory_delta_summary.csv"
    delta_cols = ["iso3c", "country", "income_group",
                  "label_code", "label_name",
                  "gini_coverage", "delta_gini", "delta_tert",
                  "delta_comp", "label_reason"]
    labels[delta_cols].to_csv(p2, index=False)
    print(f"  [OK] {p2.name}")

    # Temporal split
    p3 = SPLIT_DIR / f"temporal_split.csv"
    temporal.to_csv(p3, index=False)
    train_n = (temporal["split"] == "train").sum()
    test_n  = (temporal["split"] == "test").sum()
    print(f"  [OK] {p3.name}  "
          f"(train={train_n} rows [{YEAR_MIN}–{TRAIN_END}], "
          f"test={test_n} rows [{TEST_START}–{YEAR_MAX}])")

    # LOCO splits
    p4 = SPLIT_DIR / f"loco_splits.csv"
    loco.to_csv(p4, index=False)
    n_folds = loco["fold"].nunique()
    print(f"  [OK] {p4.name}  ({n_folds} folds, "
          f"{len(loco):,} rows)")

    print(f"\n[done] Data     : {DATA_DIR.resolve()}")
    print(f"[done] Splits   : {SPLIT_DIR.resolve()}")
    print(f"[done] Figures  : {FIGDIR.resolve()}")
    print("\n[next] Run: python src/04_descriptive_statistics.py")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    panel = load_panel()

    print("\n[step 1] Computing decade-long deltas …")
    deltas = compute_deltas(panel)

    print("\n[step 2] Assigning trajectory labels …")
    labels = assign_labels(deltas)
    print_label_summary(labels)

    print("\n[step 3] Building train/test splits …")
    temporal = build_temporal_split(panel)
    loco     = build_loco_splits(panel)
    print(f"  Temporal : {(temporal['split']=='train').sum()} train  "
          f"/ {(temporal['split']=='test').sum()} test rows")
    print(f"  LOCO     : {loco['fold'].nunique()} folds  "
          f"× {panel['year'].nunique()} years  "
          f"× 12 countries")

    print("\n[step 4] Generating figures …")
    plot_trajectory_scatter(deltas, labels)
    plot_split_overview(temporal, loco, panel)

    save_outputs(labels, temporal, loco)


if __name__ == "__main__":
    main()